In [1]:
!python --version
!which python

Python 3.12.3
/home/floriane/Documents/visual_code/.latinCy12/bin/python


importer ovide

In [2]:

import re
import requests
import numpy as np
from bs4 import BeautifulSoup
from sklearn.metrics.pairwise import cosine_similarity
import spacy

download and parse metamorphosis

In [3]:
METAMORPHOSES_URL = (
    "https://raw.githubusercontent.com/PerseusDL/canonical-latinLit/"
    "master/data/phi0959/phi006/phi0959.phi006.perseus-lat2.xml"
)
 
def download_metamorphoses(url: str = METAMORPHOSES_URL) -> str:
    """Download the XML and return clean Latin text."""
    print("Downloading Metamorphoses from Perseus Digital Library...")
    response = requests.get(url, timeout=30)
    response.raise_for_status()
 
    #soup = BeautifulSoup(response.text, "lxml-xml")
    #soup = BeautifulSoup(response.text, "xml")
    soup = BeautifulSoup(response.text, "html.parser")
 
    # Remove notes, apparatus, and non-text tags
    for tag in soup.find_all(["note", "bibl", "ref", "milestone"]):
        tag.decompose()
 
    # Extract text from <l> (verse line) elements
    lines = soup.find_all("l")
    if lines:
        text = " ".join(l.get_text(separator=" ") for l in lines)
    else:
        # Fallback: get all text
        text = soup.get_text(separator=" ")
 
    # Clean whitespace
    text = re.sub(r"\s+", " ", text).strip()
    print(f"  → Extracted {len(text):,} characters, ~{len(text.split()):,} words.\n")
    return text
 
 
def split_into_chunks(text: str, chunk_size: int = 300) -> list[str]:
    """
    Split text into chunks of roughly `chunk_size` words.
    Smaller chunks = finer-grained comparison.
    """
    words = text.split()
    return [
        " ".join(words[i : i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]
 

In [ ]:
# Step 1 — Download Metamorphoses
ovid_full = download_metamorphoses()

# Save to .txt file
with open("metamorphoses.txt", "w", encoding="utf-8") as f:
    f.write(ovid_full)
print("Saved to metamorphoses.txt")

In [4]:
def load_model(model_name: str = "la_core_web_lg"):
    print(f"Loading LatinCy model '{model_name}'...")
    nlp = spacy.load(model_name)
    print(f"  → Model loaded. Vector size: {nlp.vocab.vectors_length}\n")
    return nlp

In [5]:

def vectorise(nlp, texts: list[str]) -> np.ndarray:
    """Return a (N, vector_dim) matrix of mean-token vectors."""
    vectors = []
    for doc in nlp.pipe(texts, batch_size=32, disable=["ner", "parser"]):
        vectors.append(doc.vector)
    return np.array(vectors)

In [6]:
 
def find_most_similar(
    my_text: str,
    ovid_chunks: list[str],
    ovid_vectors: np.ndarray,
    nlp,
    top_n: int = 5,
) -> list[dict]:
    """
    Compare `my_text` against all Ovid chunks.
    Returns the top_n most similar passages.
    """
    my_vec = nlp(my_text).vector.reshape(1, -1)
    scores = cosine_similarity(my_vec, ovid_vectors)[0]
 
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_n]
 
    results = []
    for idx, score in ranked:
        results.append({
            "rank": len(results) + 1,
            "score": score,
            "chunk_index": idx,
            "text_preview": ovid_chunks[idx][:200] + "...",
        })
    return results
 
 
def word_level_similarity(
    my_text: str,
    ovid_text: str,
    nlp,
    threshold: float = 0.80,
) -> list[dict]:
    """
    Find token pairs between your text and a passage of Ovid
    with similarity above `threshold`.
    """
    doc1 = nlp(my_text)
    doc2 = nlp(ovid_text)
 
    pairs = []
    for t1 in doc1:
        if not t1.has_vector or t1.is_stop or t1.is_punct:
            continue
        for t2 in doc2:
            if not t2.has_vector or t2.is_stop or t2.is_punct:
                continue
            sim = t1.similarity(t2)
            if sim >= threshold:
                pairs.append({
                    "your_word": t1.text,
                    "ovid_word": t2.text,
                    "similarity": round(sim, 4),
                })
 
    # Deduplicate and sort
    seen = set()
    unique_pairs = []
    for p in sorted(pairs, key=lambda x: -x["similarity"]):
        key = (p["your_word"], p["ovid_word"])
        if key not in seen:
            seen.add(key)
            unique_pairs.append(p)
 
    return unique_pairs[:30]  # return top 30

In [7]:
with open("De_somniis/De_somniis.txt", "r", encoding="utf-8") as f:
    MY_TEXT = f.read()
# ↑ Replace this with your own Latin text

In [8]:
    # Step 1 — Download Metamorphoses
    ovid_full = download_metamorphoses()
 
    # Optionally save locally so you don't re-download every run
    with open("metamorphoses.txt", "w", encoding="utf-8") as f:
        f.write(ovid_full)
    print("Saved to metamorphoses.txt\n")

/tmp/ipykernel_34090/54871623.py:14: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(response.text, "html.parser")


  → Extracted 539,103 characters, ~77,677 words.

Saved to metamorphoses.txt



In [9]:

 
 
if __name__ == "__main__":
 
    # Step 1 — Download Metamorphoses
    ovid_full = download_metamorphoses()
 
    # Optionally save locally so you don't re-download every run
    with open("metamorphoses.txt", "w", encoding="utf-8") as f:
        f.write(ovid_full)
    print("Saved to metamorphoses.txt\n")
 
    # Step 2 — Load model
    nlp = load_model("la_core_web_lg")
 
    # Step 3 — Split Ovid into chunks and vectorise
    print("Splitting Metamorphoses into chunks and computing vectors...")
    chunks = split_into_chunks(ovid_full, chunk_size=300)
    print(f"  → {len(chunks)} chunks of ~300 words each.")
    ovid_vecs = vectorise(nlp, chunks)
    print(f"  → Vector matrix shape: {ovid_vecs.shape}\n")
 
    # Step 4a — Document-level similarity (top 5 passages)
    print("=" * 60)
    print("TOP 5 MOST SIMILAR PASSAGES IN THE METAMORPHOSES")
    print("=" * 60)
    results = find_most_similar(MY_TEXT, chunks, ovid_vecs, nlp, top_n=5)
    for r in results:
        print(f"\nRank #{r['rank']}  |  Score: {r['score']:.4f}  |  Chunk #{r['chunk_index']}")
        print(r["text_preview"])
 
    # Step 4b — Word-level similarity against the best passage
    best_chunk = chunks[results[0]["chunk_index"]]
    print("\n" + "=" * 60)
    print("WORD-LEVEL SIMILARITIES (threshold ≥ 0.80)")
    print("vs. best matching passage")
    print("=" * 60)
    word_pairs = word_level_similarity(MY_TEXT, best_chunk, nlp, threshold=0.80)
    if word_pairs:
        for p in word_pairs:
            print(f"  {p['your_word']:20s} ↔  {p['ovid_word']:20s}  ({p['similarity']})")
    else:
        print("  No word pairs above threshold found.")
 
    # Step 4c — Overall single similarity score
    doc_my = nlp(MY_TEXT)
    doc_ovid = nlp(ovid_full[:5000])  # first 5000 chars as a representative sample
    print(f"\nOverall document similarity (your text vs. Metamorphoses sample): "
          f"{doc_my.similarity(doc_ovid):.4f}")

/tmp/ipykernel_34090/54871623.py:14: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(response.text, "html.parser")


  → Extracted 539,103 characters, ~77,677 words.

Saved to metamorphoses.txt

Loading LatinCy model 'la_core_web_lg'...
  → Model loaded. Vector size: 300

Splitting Metamorphoses into chunks and computing vectors...
  → 259 chunks of ~300 words each.
  → Vector matrix shape: (259, 300)

TOP 5 MOST SIMILAR PASSAGES IN THE METAMORPHOSES

Rank #1  |  Score: 0.8249  |  Chunk #256
de Latia pinu Phoebeius anguis contulit et finem, specie caeleste resumpta, luctibus imposuit venitque salutifer urbi. Hic tamen accessit delubris advena nostris: Caesar in urbe sua deus est; quem Mar...

Rank #2  |  Score: 0.8205  |  Chunk #228
pocula parte sumpta mihi fuerant, illa vestigia feci, cumque eadem passis (tantum medicamina possunt!) claudor hara, solumque suis caruisse figura vidimus Eurylochum: solus data pocula fugit. Quae nis...

Rank #3  |  Score: 0.8181  |  Chunk #250
et quidquid in illa est: nos quoque, pars mundi, quoniam non corpora solum, verum etiam volucres animae sumus inque ferinas poss

In [10]:
def word_level_similarity(
    my_text: str,
    ovid_passages: list[str],   # now accepts multiple passages
    nlp,
    threshold: float = 0.70,    # lowered from 0.80
) -> list[dict]:
    """
    Find token pairs between your text and a list of Ovid passages.
    Uses lemmas instead of raw word forms.
    """
    doc1 = nlp(my_text)
    pairs = []

    for passage in ovid_passages:
        doc2 = nlp(passage)
        for t1 in doc1:
            if not t1.has_vector or t1.is_stop or t1.is_punct:
                continue
            for t2 in doc2:
                if not t2.has_vector or t2.is_stop or t2.is_punct:
                    continue
                sim = t1.similarity(t2)
                if sim >= threshold:
                    pairs.append({
                        "your_word":  t1.text,
                        "your_lemma": t1.lemma_,   # e.g. "iactatus" → "iacto"
                        "ovid_word":  t2.text,
                        "ovid_lemma": t2.lemma_,
                        "similarity": round(sim, 4),
                    })

    # Deduplicate on lemma pairs
    seen = set()
    unique_pairs = []
    for p in sorted(pairs, key=lambda x: -x["similarity"]):
        key = (p["your_lemma"], p["ovid_lemma"])
        if key not in seen:
            seen.add(key)
            unique_pairs.append(p)

    return unique_pairs[:50]

In [11]:
# Collect the top 5 passage texts
top_passages = [chunks[r["chunk_index"]] for r in results]

print("\n" + "=" * 60)
print("WORD-LEVEL SIMILARITIES (threshold ≥ 0.70)")
print("vs. top 5 matching passages, grouped by lemma")
print("=" * 60)
word_pairs = word_level_similarity(MY_TEXT, top_passages, nlp, threshold=0.70)

for p in word_pairs:
    print(f"  {p['your_word']:15s} ({p['your_lemma']:15s})"
          f" ↔  {p['ovid_word']:15s} ({p['ovid_lemma']:15s})"
          f"  sim={p['similarity']}")


WORD-LEVEL SIMILARITIES (threshold ≥ 0.70)
vs. top 5 matching passages, grouped by lemma
  que             (que            ) ↔  que             (que            )  sim=1.0
  virum           (uir            ) ↔  virum           (uir            )  sim=1.0
  erat            (sum            ) ↔  erat            (sum            )  sim=1.0
  scilicet        (scilicet       ) ↔  scilicet        (scilicet       )  sim=1.0
  quod            (quod           ) ↔  quod            (quod           )  sim=1.0
  ut              (ut             ) ↔  ut              (ut             )  sim=1.0
  solum           (solum          ) ↔  solum           (solum          )  sim=1.0
  opus            (opus           ) ↔  opus            (opus           )  sim=1.0
  mei             (ego            ) ↔  mei             (meus           )  sim=1.0
  plus            (plus           ) ↔  plus            (plus           )  sim=1.0
  que             (que            ) ↔  que             (qui            )  sim=1.0
  vestig

In [12]:
from itertools import islice

def make_ngrams(doc, n: int) -> list[dict]:
    """Extract n-grams from a spaCy doc, skipping punct/space tokens."""
    tokens = [t for t in doc if not t.is_punct and not t.is_space]
    ngrams = []
    for i in range(len(tokens) - n + 1):
        chunk = tokens[i:i+n]
        ngrams.append({
            "text": " ".join(t.text for t in chunk),
            "lemma": " ".join(t.lemma_ for t in chunk),
            # mean vector of the n-gram
            "vector": np.mean([t.vector for t in chunk], axis=0),
        })
    return ngrams


def word_and_ngram_similarity(
    my_text: str,
    ovid_passages: list[str],
    nlp,
    word_threshold: float = 0.70,   # lowered
    ngram_threshold: float = 0.82,  # ngrams need a higher bar (more specific)
    ngram_sizes: list[int] = [2, 3], # bigrams and trigrams
) -> dict:
    """
    Combined word-level + n-gram similarity between your text and Ovid passages.
    Returns separate lists for words and ngrams.
    """
    doc1 = nlp(my_text)
    my_ngrams = {n: make_ngrams(doc1, n) for n in ngram_sizes}

    word_pairs = []
    ngram_pairs = []

    for passage in ovid_passages:
        doc2 = nlp(passage)

        # ── Word-level ──────────────────────────────────────────
        for t1 in doc1:
            if not t1.has_vector or t1.is_stop or t1.is_punct:
                continue
            for t2 in doc2:
                if not t2.has_vector or t2.is_stop or t2.is_punct:
                    continue
                sim = t1.similarity(t2)
                if sim >= word_threshold:
                    word_pairs.append({
                        "your_word":  t1.text,
                        "your_lemma": t1.lemma_,
                        "ovid_word":  t2.text,
                        "ovid_lemma": t2.lemma_,
                        "similarity": round(sim, 4),
                    })

        # ── N-gram level ─────────────────────────────────────────
        for n in ngram_sizes:
            ovid_ngrams = make_ngrams(doc2, n)
            for ng1 in my_ngrams[n]:
                if ng1["vector"] is None:
                    continue
                for ng2 in ovid_ngrams:
                    if ng2["vector"] is None:
                        continue
                    v1 = ng1["vector"].reshape(1, -1)
                    v2 = ng2["vector"].reshape(1, -1)
                    sim = cosine_similarity(v1, v2)[0][0]
                    if sim >= ngram_threshold:
                        ngram_pairs.append({
                            "n": n,
                            "your_ngram":  ng1["text"],
                            "your_lemma":  ng1["lemma"],
                            "ovid_ngram":  ng2["text"],
                            "ovid_lemma":  ng2["lemma"],
                            "similarity":  round(float(sim), 4),
                        })

    # Deduplicate on lemma pairs
    def dedup(pairs, key_fields):
        seen, out = set(), []
        for p in sorted(pairs, key=lambda x: -x["similarity"]):
            key = tuple(p[k] for k in key_fields)
            if key not in seen:
                seen.add(key)
                out.append(p)
        return out

    return {
        "words":  dedup(word_pairs,  ["your_lemma", "ovid_lemma"])[:50],
        "ngrams": dedup(ngram_pairs, ["your_lemma", "ovid_lemma"])[:50],
    }

In [13]:
top_passages = [chunks[r["chunk_index"]] for r in results]

print("\n" + "=" * 60)
print("WORD-LEVEL SIMILARITIES (threshold ≥ 0.70)")
print("=" * 60)
matches = word_and_ngram_similarity(MY_TEXT, top_passages, nlp,
                                    word_threshold=0.70,
                                    ngram_threshold=0.82)

for p in matches["words"]:
    print(f"  {p['your_word']:15s} ({p['your_lemma']:12s})"
          f" ↔  {p['ovid_word']:15s} ({p['ovid_lemma']:12s})"
          f"  sim={p['similarity']}")

print("\n" + "=" * 60)
print("N-GRAM SIMILARITIES (bigrams & trigrams, threshold ≥ 0.82)")
print("=" * 60)
for p in matches["ngrams"]:
    label = f"{'bigram' if p['n']==2 else 'trigram'}"
    print(f"  [{label}]  '{p['your_ngram']}'"
          f"  ↔  '{p['ovid_ngram']}'"
          f"  sim={p['similarity']}")


WORD-LEVEL SIMILARITIES (threshold ≥ 0.70)
  que             (que         ) ↔  que             (que         )  sim=1.0
  virum           (uir         ) ↔  virum           (uir         )  sim=1.0
  erat            (sum         ) ↔  erat            (sum         )  sim=1.0
  scilicet        (scilicet    ) ↔  scilicet        (scilicet    )  sim=1.0
  quod            (quod        ) ↔  quod            (quod        )  sim=1.0
  ut              (ut          ) ↔  ut              (ut          )  sim=1.0
  solum           (solum       ) ↔  solum           (solum       )  sim=1.0
  opus            (opus        ) ↔  opus            (opus        )  sim=1.0
  mei             (ego         ) ↔  mei             (meus        )  sim=1.0
  plus            (plus        ) ↔  plus            (plus        )  sim=1.0
  que             (que         ) ↔  que             (qui         )  sim=1.0
  vestigia        (uestigium   ) ↔  vestigia        (uestigium   )  sim=1.0
  tempore         (tempus      ) ↔  tempore 